# Remaining Useful Life Prediction with Weibull Analysis

This notebook demonstrates the **WeibullRULPredictor** for reliability analysis:

1. Fit a Weibull distribution to time-to-failure data (with censoring)
2. Plot the survival curve and hazard rate
3. Predict RUL with confidence intervals
4. Visualize the Weibull PDF

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from anomalykit import WeibullRULPredictor
from generate_data import generate_failure_data

## Data Generation

Generate 200 component lifetimes from a Weibull distribution (shape=2.5, scale=1500 h) with ~30% right-censoring.

In [ ]:
df = generate_failure_data(n=200, seed=42)

print(f"Total components: {len(df)}")
print(f"Failed: {df['failed'].sum()}  |  Censored: {(~df['failed']).sum()}")
print(f"\nFailure mode distribution:")
print(df["failure_mode"].value_counts())
df.head(10)

## Fit Weibull Model

We fit two models: one using only uncensored data (simple) and one using the full censored dataset (MLE).

In [ ]:
# Model 1: uncensored data only
failed_hours = df.loc[df["failed"], "hours"].values
predictor_simple = WeibullRULPredictor().fit(failed_hours)
print("Simple model (uncensored only):")
print(f"  Parameters: {predictor_simple.get_parameters()}")

# Model 2: with censoring information
all_hours = df["hours"].values
censored = ~df["failed"].values
predictor_censored = WeibullRULPredictor().fit(all_hours, censored=censored)
print("\nCensored model (MLE):")
print(f"  Parameters: {predictor_censored.get_parameters()}")

## Survival Curve

In [ ]:
t_range = np.linspace(1, 3000, 500)

_, surv_simple = predictor_simple.get_survival_curve(t_range)
_, surv_censored = predictor_censored.get_survival_curve(t_range)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_range, surv_simple, lw=2, label=f"Simple (shape={predictor_simple.shape:.2f}, scale={predictor_simple.scale:.0f})")
ax.plot(t_range, surv_censored, lw=2, ls="--", label=f"Censored MLE (shape={predictor_censored.shape:.2f}, scale={predictor_censored.scale:.0f})")
ax.axhline(0.5, color="gray", ls=":", lw=0.8)
ax.axvline(predictor_censored.median_life, color="gray", ls=":", lw=0.8, label=f"Median life = {predictor_censored.median_life:.0f} h")
ax.set_xlabel("Operating Hours")
ax.set_ylabel("Survival Probability S(t)")
ax.set_title("Weibull Survival Curve")
ax.legend()
ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Hazard Rate

In [ ]:
_, hazard = predictor_censored.get_hazard_curve(t_range)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_range, hazard, lw=2, color="tomato")
ax.set_xlabel("Operating Hours")
ax.set_ylabel("Hazard Rate h(t)")
ax.set_title(f"Weibull Hazard Rate (shape={predictor_censored.shape:.2f} > 1 = wear-out)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## RUL Prediction with Confidence Intervals

In [ ]:
current_hours_list = [200, 500, 800, 1000, 1200, 1400]

print(f"{'Current Hours':>14} {'Predicted RUL':>14} {'95% CI':>22} {'Survival':>10} {'Hazard':>10}")
print("-" * 75)

rul_results = []
for h in current_hours_list:
    r = predictor_censored.predict(h, confidence_level=0.95)
    rul_results.append(r)
    print(f"{h:>14.0f} {r.predicted_rul:>14.0f} [{r.confidence_interval[0]:>8.0f}, {r.confidence_interval[1]:>8.0f}] "
          f"{r.survival_probability:>10.3f} {r.hazard_rate:>10.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ruls = [r.predicted_rul for r in rul_results]
ci_low = [r.confidence_interval[0] for r in rul_results]
ci_high = [r.confidence_interval[1] for r in rul_results]

ax.plot(current_hours_list, ruls, "o-", lw=2, markersize=8, color="steelblue", label="Predicted RUL")
ax.fill_between(current_hours_list, ci_low, ci_high, alpha=0.2, color="steelblue", label="95% CI")
ax.set_xlabel("Current Operating Hours")
ax.set_ylabel("Remaining Useful Life (hours)")
ax.set_title("RUL Prediction with Confidence Intervals")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Weibull PDF

In [ ]:
from scipy import stats

pdf_values = stats.weibull_min.pdf(t_range, predictor_censored.shape, scale=predictor_censored.scale)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_range, pdf_values, lw=2, color="darkorange", label="Weibull PDF")
ax.hist(failed_hours, bins=30, density=True, alpha=0.3, color="steelblue", edgecolor="white", label="Observed failures")
ax.axvline(predictor_censored.mean_life, color="red", ls="--", lw=1, label=f"Mean life = {predictor_censored.mean_life:.0f} h")
ax.axvline(predictor_censored.median_life, color="green", ls="--", lw=1, label=f"Median life = {predictor_censored.median_life:.0f} h")
ax.set_xlabel("Hours to Failure")
ax.set_ylabel("Density")
ax.set_title("Weibull Probability Density Function vs Observed Failures")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

- **Shape > 1** indicates wear-out failure mode (hazard rate increases with time)
- **Censored MLE** produces more accurate parameters when not all components have failed
- **RUL decreases** as operating hours increase, with narrowing confidence intervals
- The Weibull PDF fits the observed failure distribution well